In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

df_org = pd.read_csv("./house_train.csv")
df = df_org.copy()
df.head()

In [ ]:
print(df.shape)
print(df.dtypes.value_counts())
isnull_series = df.isnull().sum()
print(isnull_series[isnull_series>0].sort_values(ascending = False))

In [ ]:
df["FireplaceQu"]

In [ ]:
plt.title("Sale price Histogram")
plt.xticks(rotation = 15)
sns.histplot(df [ "SalePrice"], kde = True)
plt.show()

In [ ]:
plt.title("logtransform Sale price Histogram")
log_SalePrice = np.log1p(df["SalePrice"])
sns.histplot(log_SalePrice, kde = True)
plt.show()

In [ ]:
org_SalePrice = df["SalePrice"]
df["Saleprice"] = np.log1p(df["SalePrice"])

df.drop(["Id" , "PoolQC", "MiscFeature", "Alley", "Fence", "FireplaceQu"],axis = 1, inplace = True)

df.fillna(df.mean(), inplace=True)

null_column_count = df.isnull().sum()[df.isnull().sum() > 0]
print(df.dtypes[null_column_count.index])

In [ ]:
print(df.shape)
df_ohe = pd.get_dummies(df)
print(df_ohe.shape)

In [ ]:
null_column_count = df_ohe.isnull().sum()[df_ohe.isnull().sum()>0]

print(df_ohe.dtypes[null_column_count.index])

In [ ]:
def get_rmse(model) : 
    pred = model.predict(X_test)
    mse = mean_squared_error(y_test,pred)
    rmse = np.sqrt(mse)
    print(model.__class__.__name__, np.round(rmse,3))
    return rmse

def get_rmses(models):
    rmses = []
    for model in models:
        rmse = get_rmse(model)
        rmses.append(rmse)

    return rmses

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error

y_target = df_ohe["SalePrice"]
X_features = df_ohe.drop(["SalePrice"],axis = 1 , inplace= False)

X_train , X_test , y_train , y_test  = train_test_split(X_features, y_target, test_size= 0.2, random_state=156 )

lr_reg = LinearRegression()
lr_reg.fit(X_train,y_train)

ridge_reg = Ridge()
ridge_reg.fit(X_train, y_train)

lasso_reg = Lasso()
lasso_reg.fit(X_train, y_train)

models = [lr_reg , ridge_reg , lasso_reg]
get_rmses(models)

In [ ]:
def get_top_bottom_coef(model):
    coef = pd.Series(model.coef_, index = X_features.columns)
    coef_high = coef.sort_values(ascending= False).head(10)
    coef_low = coef.sort_values(ascending= False).tail(10)
    return coef_high , coef_low

In [ ]:
def visualize_coef(models):
    fig, axs = plt.subplots(figsize = (24, 10), nrows = 1, ncols = 3)
    fig.tight_layout()
    for i_num, model in enumerate(models):
        coef_high, coef_low = get_top_bottom_coef(model)
        coef_concat = pd.concat([coef_high, coef_low])
        axs[i_num].set_title(model.__class__.__name__+' Coeffiecents',
                             size = 25)
        axs[i_num].tick_params(axis = 'y', direction = 'in', pad = -120)
        for label in (axs[i_num].get_xticklabels() + axs[i_num].get_yticklabels()):
            label.set_fontsize(22)
        sns.barplot(x = coef_concat.values, y = coef_concat.index,
                    ax = axs[i_num])

model = [lr_reg, ridge_reg, lasso_reg]
visualize_coef(models)

In [ ]:
from sklearn.model_selection import cross_val_score

def get_avg_rmse_cv(models):
    for model in models :
        rmse_list = np.sqrt(-cross_val_score(model,X_features,y_target,scoring= "neg_mean_squared_error",cv = 5))    
        rmse_avg = np.mean(rmse_list)
        print(model.__class__.__name__, np.round(rmse_list, 3))
        print(model.__class__.__name__, np.round(rmse_avg, 3))

models = [lr_reg, ridge_reg, lasso_reg]
get_avg_rmse_cv(models)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV

raw_data = df_org.drop(
    columns=["Id", "PoolQC", "MiscFeature", "Alley", "Fence", "FireplaceQu"],
    errors="ignore"
).copy()
y_target = np.log1p(raw_data.pop("SalePrice"))
X_features = raw_data

X_train, X_test, y_train, y_test = train_test_split(
    X_features, y_target, test_size=0.2, random_state=156
)


In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(df_org["GrLivArea"], df_org["SalePrice"], alpha=0.5)
plt.xlabel("Above ground living area (square feet)")
plt.ylabel("Sale price")
plt.title("Living area and sale price")
plt.show()

numeric_train = X_train.select_dtypes(include="number")
skew_values = numeric_train.skew().sort_values(ascending=False)
print(skew_values.abs().sort_values(ascending=False).head(15))


In [ ]:
numeric_features = X_train.select_dtypes(include="number").columns
categorical_features = X_train.select_dtypes(exclude="number").columns

numeric_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_preprocessor, numeric_features),
    ("categorical", categorical_preprocessor, categorical_features)
])

def report_rmse(name, model, X, y_log):
    pred_log = model.predict(X)
    rmse_log = np.sqrt(mean_squared_error(y_log, pred_log))
    pred_price = np.expm1(pred_log)
    true_price = np.expm1(y_log)
    rmse_price = np.sqrt(mean_squared_error(true_price, pred_price))
    print(f"{name} | log RMSE: {rmse_log:,.4f} | price RMSE: ${rmse_price:,.0f}")
    return rmse_log, rmse_price

linear_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])
linear_model.fit(X_train, y_train)
report_rmse("LinearRegression", linear_model, X_test, y_test)


In [ ]:
ridge_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", Ridge())
])
lasso_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", Lasso(max_iter=20000))
])

ridge_params = {"model__alpha": [0.05, 0.1, 1, 5, 8, 10, 12, 15, 20]}
lasso_params = {"model__alpha": [0.0001, 0.001, 0.005, 0.008, 0.03, 0.05, 0.1, 0.5, 1]}

ridge_search = GridSearchCV(
    ridge_pipeline, ridge_params, scoring="neg_mean_squared_error",
    cv=5, n_jobs=-1, refit=True
)
lasso_search = GridSearchCV(
    lasso_pipeline, lasso_params, scoring="neg_mean_squared_error",
    cv=5, n_jobs=-1, refit=True
)
ridge_search.fit(X_train, y_train)
lasso_search.fit(X_train, y_train)

best_ridge = ridge_search.best_estimator_
best_lasso = lasso_search.best_estimator_
print(ridge_search.best_params_)
print("Ridge CV RMSE:", np.sqrt(-ridge_search.best_score_).round(4))
print(lasso_search.best_params_)
print("Lasso CV RMSE:", np.sqrt(-lasso_search.best_score_).round(4))


In [ ]:
report_rmse("LinearRegression", linear_model, X_test, y_test)
report_rmse("Ridge", best_ridge, X_test, y_test)
report_rmse("Lasso", best_lasso, X_test, y_test)


In [ ]:
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

xgb_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBRegressor(
        n_estimators=600, learning_rate=0.05,
        colsample_bytree=0.5, subsample=0.8,
        objective="reg:squarederror", random_state=156, n_jobs=-1
    ))
])
lgbm_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LGBMRegressor(
        n_estimators=600, learning_rate=0.05, num_leaves=4,
        subsample=0.6, colsample_bytree=0.4,
        reg_lambda=10, verbosity=-1, random_state=156, n_jobs=-1
    ))
])

xgb_model.fit(X_train, y_train)
lgbm_model.fit(X_train, y_train)
report_rmse("XGBoost", xgb_model, X_test, y_test)
report_rmse("LightGBM", lgbm_model, X_test, y_test)


In [ ]:
models = {
    "Ridge": best_ridge,
    "Lasso": best_lasso,
    "XGBoost": xgb_model,
    "LightGBM": lgbm_model
}
ensemble_pred_log = np.mean(
    [model.predict(X_test) for model in models.values()], axis=0
)
ensemble_log_rmse = np.sqrt(mean_squared_error(y_test, ensemble_pred_log))
ensemble_price_rmse = np.sqrt(mean_squared_error(
    np.expm1(y_test), np.expm1(ensemble_pred_log)
))
print("Ensemble log RMSE:", round(ensemble_log_rmse, 4))
print("Ensemble price RMSE:", round(ensemble_price_rmse, 0))


In [ ]:
from sklearn.ensemble import StackingRegressor

stacking_model = StackingRegressor(
    estimators=[
        ("ridge", best_ridge),
        ("lasso", best_lasso),
        ("xgb", xgb_model),
        ("lgbm", lgbm_model)
    ],
    final_estimator=Ridge(alpha=1.0),
    cv=5,
    n_jobs=-1
)
stacking_model.fit(X_train, y_train)
report_rmse("Stacking", stacking_model, X_test, y_test)
